In [ ]:
# 1. 필수 라이브러리 및 LLaMA-Factory 설치
# (최신 버전 충돌 방지를 위해 transformers 버전을 4.57.1로 고정합니다)
!git clone https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory

# 의존성 지옥 방지용 강제 버전 고정 설치
!pip install "transformers==4.57.1" "huggingface_hub<1.0.0"
!pip install -e .[metrics]
!pip install -q qwen_vl_utils bitsandbytes scipy
!pip install flash-attn --no-build-isolation

# 필수 라이브러리 설치
!pip install faker tqdm

print("✅ 환경 설정 완료! [런타임] -> [세션 다시 시작]을 누르고 2단계로 넘어가세요.")

Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 26159, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 26159 (delta 23), reused 39 (delta 12), pack-reused 26069 (from 4)
Receiving objects: 100% (26159/26159), 12.65 MiB | 17.82 MiB/s, done.
Resolving deltas: 100% (18796/18796), done.
/content/LLaMA-Factory
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.3/566.3 kB 46.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.3.4
    Uninstalling huggingface_hub-1.3.4:
      Successfully uninstalled huggingface_hub-1.3.4
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
Obtaining

In [ ]:
import os
import json
import random
import requests
import shutil
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from faker import Faker
from tqdm import tqdm

# --- 설정 ---
BASE_DIR = "/content"
DATA_DIR = os.path.join(BASE_DIR, "data_vlm")
TEMPLATE_PATH = Path("/content/waybill_template.jpg")
FONTS_DIR = Path("/content/fonts")
GENERATE_COUNT = 1000  # 3만장은 너무 많으니 5000장으로 테스트 (충분함)

# --- 0. 환경 초기화 ---
if os.path.exists(DATA_DIR): shutil.rmtree(DATA_DIR)
os.makedirs(os.path.join(DATA_DIR, "images"), exist_ok=True)
if not FONTS_DIR.exists(): FONTS_DIR.mkdir()

# --- 1. 폰트 다운로드 ---
FONT_URLS = {
    "NanumGothicBold": "https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Bold.ttf",
    "DoHyeon": "https://github.com/google/fonts/raw/main/ofl/dohyeon/DoHyeon-Regular.ttf",
}
for name, url in FONT_URLS.items():
    if not (FONTS_DIR / f"{name}.ttf").exists():
        try:
            r = requests.get(url)
            with open(FONTS_DIR / f"{name}.ttf", 'wb') as f: f.write(r.content)
        except: pass

def get_font():
    fonts = list(FONTS_DIR.glob("*.ttf"))
    return str(random.choice(fonts)) if fonts else "arial.ttf"

# --- 2. 데이터 생성기 ---
fake = Faker('ko_KR')

class VLMDataGenerator:
    def __init__(self):
        if not TEMPLATE_PATH.exists():
            raise FileNotFoundError("❌ waybill_template.jpg 파일이 없습니다! 경로를 확인해주세요.")
        self.bg = Image.open(TEMPLATE_PATH).convert("RGB")

        # ✅ [수정됨] 요청하신 정확한 좌표값 적용 (x1, y1, x2, y2)
        self.positions = {
            'tracking_number': (16, 191, 218, 241),    # 운송장번호
            'region_code': (443, 170, 628, 238),       # 분류코드
            'recipient_name': (60, 258, 200, 312),     # 받는분 이름
            'recipient_address': (15, 320, 630, 375),  # 받는분 주소
            'sender_name': (92, 394, 221, 454),        # 보내는분 이름
            'sender_address': (12, 458, 632, 537)      # 보내는분 주소
        }

    def render(self):
        img = self.bg.copy()
        draw = ImageDraw.Draw(img)

        # ✅ [수정됨] 영문 키로 데이터 생성
        data = {
            "tracking_number": ''.join([str(random.randint(0, 9)) for _ in range(12)]),
            "region_code": f"{fake.city()[:2]}{random.randint(1, 9)}",
            "recipient_name": fake.name(),
            "recipient_address": fake.address(),
            "sender_name": fake.name(),
            "sender_address": fake.address()
        }

        font_path = get_font()

        for key, text in data.items():
            if key not in self.positions: continue

            x1, y1, x2, y2 = self.positions[key]

            # 폰트 크기 자동 조절 (박스 안에 맞게)
            # 시작 크기를 조금 키웠습니다 (40->50) 꽉 차게 보이도록
            size = 50
            font = ImageFont.truetype(font_path, size)

            # 박스 너비(x2-x1) 또는 높이(y2-y1)를 넘어가면 줄임
            while (font.getlength(text) > (x2 - x1) or size > (y2 - y1)) and size > 10:
                size -= 2
                font = ImageFont.truetype(font_path, size)

            # 텍스트 그리기 (약간의 여백을 위해 y1에 +2 정도 보정)
            draw.text((x1 + 5, y1 + 2), text, font=font, fill=(30, 30, 30))

        return img, data

# --- 3. 생성 실행 ---
generator = VLMDataGenerator()
llama_data = []

print(f"🚀 데이터 {GENERATE_COUNT}장 재생성 중 (영문 키 형식 적용)...")
for i in tqdm(range(GENERATE_COUNT)):
    img, json_data = generator.render()
    filename = f"{i:05d}.jpg"
    img_path = os.path.join(DATA_DIR, "images", filename)
    img.save(img_path)

    entry = {
        "messages": [
            {"role": "user", "content": "<image>이 운송장 이미지에서 정보를 추출하여 JSON 형식으로 출력해줘."},
            {"role": "assistant", "content": json.dumps(json_data, ensure_ascii=False)}
        ],
        "images": [img_path]
    }
    llama_data.append(entry)

train_file = os.path.join(DATA_DIR, "train_data.json")
with open(train_file, "w", encoding='utf-8') as f:
    json.dump(llama_data, f, indent=2, ensure_ascii=False)

# --- 4. LLaMA-Factory 등록 ---
info = {
  "waybill_vlm": {
    "file_name": train_file,
    "formatting": "sharegpt",
    "columns": {"messages": "messages", "images": "images"},
    "tags": {
        "role_tag": "role",
        "content_tag": "content",
        "user_tag": "user",
        "assistant_tag": "assistant"
    }
  }
}
info_path = "/content/LLaMA-Factory/data/dataset_info.json"
with open(info_path, "r") as f: original_info = json.load(f)
original_info.update(info)
with open(info_path, "w") as f: json.dump(original_info, f, indent=2)

print("✅ 영문 키 형식으로 데이터 생성 완료!")

In [ ]:
import json

# 올바른 설정 (role 태그를 'role'로 명시)
info = {
  "waybill_vlm": {
    "file_name": "/content/data_vlm/train_data.json",
    "formatting": "sharegpt",
    "columns": {
      "messages": "messages",
      "images": "images"
    },
    "tags": {
      "role_tag": "role",         # [수정] False -> "role"
      "content_tag": "content",   # [추가] 내용은 "content"에 있음
      "user_tag": "user",         # [추가] 유저는 "user"라고 적혀있음
      "assistant_tag": "assistant" # [추가] AI는 "assistant"라고 적혀있음
    }
  }
}

# 파일 덮어쓰기
info_path = "/content/LLaMA-Factory/data/dataset_info.json"

with open(info_path, "r") as f:
    original_info = json.load(f)


original_info.update(info)

with open(info_path, "w") as f:
    json.dump(original_info, f, indent=2)

In [ ]:
pip install torch==2.6.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 39.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 40.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 47.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 764.4/764.4 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 120.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.7/166.7 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 111.4 MB/s eta 0:00:00
  Attemp

In [ ]:
# A100 최적화 학습 (Flash Attention 없이)
%cd /content/LLaMA-Factory

!llamafactory-cli train \
    --stage sft \
    --do_train \
    --model_name_or_path Qwen/Qwen2-VL-2B-Instruct \
    --dataset waybill_vlm \
    --template qwen2_vl \
    --finetuning_type lora \
    --lora_target all \
    --output_dir /content/qwen2_vl_finetuned \
    --per_device_train_batch_size 16 \
    --gradient_accumulation_steps 2 \
    --learning_rate 1e-4 \
    --num_train_epochs 3.0 \
    --max_grad_norm 1.0 \
    --lr_scheduler_type cosine \
    --warmup_ratio 0.1 \
    --bf16 True \
    --plot_loss True \
    --logging_steps 10 \
    --save_steps 500 \
    --save_total_limit 2 \
    --overwrite_output_dir \
    --report_to none \
    --dataloader_num_workers 4 \
    --dataloader_pin_memory True

/content/LLaMA-Factory
2026-02-02 12:29:11.583588: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-02 12:29:11.600815: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770035351.621917    4157 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770035351.628423    4157 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770035351.644693    4157 computation_placer.cc:177] computation placer already registered. Please check lin

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from peft import PeftModel
import torch
from PIL import Image

# 1. 베이스 모델 로드
print("🤖 학습된 모델을 불러오는 중입니다...")
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

# 2. LoRA 어댑터 로드 및 병합
model_path = "/content/qwen2_vl_finetuned"
model = PeftModel.from_pretrained(base_model, model_path)
model = model.merge_and_unload()  # 이제 정상 작동

processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

# 3. 테스트할 이미지
test_img_path = "/content/테스트3.png"
image = Image.open(test_img_path)
print("📸 테스트 이미지:")
display(image.resize((400, 250)))

# 4. 질문 던지기
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": test_img_path},
            {"type": "text", "text": "이 운송장 이미지에서 정보를 추출하여 JSON 형식으로 출력해줘."},
        ],
    }
]

# 5. 추론 실행
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to("cuda")

print("⏳ AI가 분석 중...")
generated_ids = model.generate(**inputs, max_new_tokens=512)
generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

print("\n🎉 [최종 결과]:")
print(output_text[0])

🤖 학습된 모델을 불러오는 중입니다...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

FileNotFoundError: [Errno 2] No such file or directory: '/content/테스트3.png'

In [ ]:
import shutil

# 모델 폴더를 zip으로 압축
shutil.make_archive('/content/qwen2_vl_finetuned', 'zip', '/content/qwen2_vl_finetuned')

# Colab에서 다운로드
from google.colab import files
files.download('/content/qwen2_vl_finetuned.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>